In [ ]:
import numpy as np
from joblib import load
from loguru import logger
from torch import Tensor
from transformers import BartForConditionalGeneration, BertForSequenceClassification
import plotly.io as pio
pio.renderers.default = "svg" # uncomment this if want interactive renderer
from carl.environment.sokoban.tokenizer import SokobanTokenizer
from carl.environment.sokoban.env import SokobanEnv

from carl.inference_components.conditional_low_level_policy import (
    ConditionalLowLevelPolicy,
    TransformerConditionalLowLevelPolicy,
)
from carl.inference_components.policy import Policy, TransformerPolicy
from carl.inference_components.subgoal_generator import (
    SubgoalGenerator,
    TransformerSubgoalGenerator,
)
from carl.inference_components.value import TransformerValue, Value

logger.info('testing inference components')

# Env and dataset

In [ ]:
import joblib as jl
datapath = './rl-data/validation/sokoban/progress/boards_1000_b4_gs100_c300_p0.35/boards_1000_b4_gs100_c300_p0.35.joblib'
data = jl.load(datapath)

env = SokobanEnv(SokobanTokenizer(None, size_of_board=(12, 12)), num_boxes=4)
env.state_to_repr(data[0])

In [ ]:
logger.info('setting up the environment')
logger.info(
    'It is important to set the categorical distance to 100, because the model was trained with this value. Also the '
    'size of the board should be (12, 12), because the model was trained with this value.'
)
env = SokobanEnv(SokobanTokenizer(cut_distance=None, size_of_board=(12, 12)), num_boxes=4)

# logger.info('setting up the dataset')
path_to_dataset: str = './rl-data/validation/sokoban/offline/12-12-4/sokoban_12_12_4_trajectories_part_88.pkl'
dataset: dict[int, list[np.ndarray]] = load(path_to_dataset)
keys = list(dataset.keys())
current_state: np.ndarray = dataset[keys[1]][1]
display(env.state_to_repr(current_state, title='current state'))
logger.info(
    'state after k is the state after k steps, where k is the number of steps the agent should take. In this '
    'case k = 3.'
)
state_after_k: np.ndarray = dataset[keys[1]][4]
display(env.many_states_to_repr([current_state, state_after_k], titles=['current state', 'state after k=3']))

In [ ]:
from carl.environment.instance_generator import (
    GeneralIterableDataLoader,
    BasicInstanceGenerator,
)

# rl-data/validation/sokoban/offline/12-12-4
path_to_folder_with_data = './rl-data/validation/sokoban/progress/boards_1000_b4_gs25_c300_p0.35'   # path to the folder with trajectories
# its a dict mapping id of trajectory to the list of states in this trajectory
env = SokobanEnv(SokobanTokenizer(None, size_of_board=(12, 12)), num_boxes=4)
instance_generator = BasicInstanceGenerator(
    generator=GeneralIterableDataLoader(path_to_folder_with_data), batch_size=32
)


initial_state_loader = iter(instance_generator.reset_dataloader())
initial_state = next(initial_state_loader).cpu().numpy()
initial_state.shape, type(initial_state), initial_state.dtype

In [ ]:
state = env.set_state(initial_state[0, :])
print(state.shape)
env.state_to_repr(state, 'Example state')

In [ ]:
dataloader = iter(instance_generator.reset_dataloader())
state = next(dataloader).cpu().numpy()[0]
state3 = next(dataloader).cpu().numpy()[0]
env.many_states_to_repr([state, state3], ['test.png', 'test3.png'])

In [ ]:
components_prefix = './rl-data/validation/sokoban/components/full_data/'
from os.path import join

path_to_policy_weights: str = join(components_prefix, 'policy/checkpoint-94820')
path_to_cllp_weights: str = join(components_prefix, 'cllp/8/checkpoint-167585')
path_to_value_function_weights: str = join(components_prefix, 'value/checkpoint-1343100')
path_to_generator_weights: str = join(components_prefix, 'generator/border/8/checkpoint-75856')

# Policy
### CurrentState $\rightarrow$ Action

In [ ]:
current_state.shape

In [ ]:
logger.info('setting up the policy')

policy: Policy = TransformerPolicy(
    policy_network_class=BertForSequenceClassification.from_pretrained,
    path_to_policy_weights=path_to_policy_weights,
    env=env,
    n_actions=2,
)
policy.construct_network()
policy_prediction: Tensor = policy.get_actions(current_state)

logger.info(f'policy prediction - distribution over action: {policy_prediction}')
logger.info(f'Proposed {len(policy_prediction)} actions')
logger.info(f'policy prediction - best action ({policy_prediction[0][0]}) with value {policy_prediction[0][1]}')

In [ ]:
# Lets visualize the policy we have learned

input_state = next(dataloader).cpu().numpy()[0]


visited_states = []
current_state = input_state
for i in range(5):
    visited_states.append(current_state.copy())
    action = policy.get_actions(current_state)
    action = action[0][0]
    logger.info(f'Action: {action}')
    env.set_state(current_state)
    state, _, done, _ = env.step(action)

    current_state = state

    if done:
        break

    if done:
        break

env.many_states_to_repr(
    visited_states, titles=[f'state_{i}' for i in range(len(visited_states) + 1)]
)

# Conditional Low Level Policy
### CurrentState $\times$ SubgoalState $\rightarrow$ Action

In [ ]:
cllp: ConditionalLowLevelPolicy = TransformerConditionalLowLevelPolicy(
    BertForSequenceClassification.from_pretrained, path_to_cllp_weights, env
)
cllp.construct_network()

In [ ]:
current_state: np.ndarray = dataset[keys[1]][1]
state_after_k: np.ndarray = dataset[keys[1]][6]

display(env.many_states_to_repr([current_state, state_after_k], ['current state', 'state after k=5']))

visited_states = []
env.core.restore_full_state_from_np_array_version(current_state)
for i in range(7):
    visited_states.append(current_state.copy())
    if np.array_equal(current_state, state_after_k):
        logger.success('Reached the goal state')
        break
    action = cllp.get_action(current_state, state_after_k)
    action = action.argmax()
    logger.info(f'Action: {action}')
    env.set_state(current_state)
    state, _, done, _ = env.step(action)

    current_state = state


env.many_states_to_repr(
    [*visited_states, state_after_k], [*list(range(len(visited_states))), 'subgoal']
)

# Value function

In [ ]:
import plotly.graph_objects as go

value_function: Value = TransformerValue(
    value_network_class=BertForSequenceClassification.from_pretrained,
    path_to_value_network_weights=path_to_value_function_weights,
    env=env,
    type_of_evaluation='regression',
)

value_function.construct_network()
# value_function_prediction: float = value_function.get_value(current_state)
# solved_state = dataset[keys[1]][-1]
# logger.info(f'value function prediction of unsolved state: {value_function_prediction}')
# logger.info(f'value function prediction of solved state: {value_function.get_value(solved_state)}')
states = [state for state in dataset[keys[11]]]
values = [value_function.get_value(state) for state in states] # can be batched too


fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(len(values))), y=values))
fig.update_layout(title='Value function predictions', xaxis_title='step', yaxis_title='value')
fig.show()

# Goal Generator

In [ ]:
from carl.solver.nodes import SearchTreeNode

logger.info('setting up the subgoal generator')
subgoal_generation_kwargs: dict[str, int] = {
    'num_beams': 8,
    'num_return_sequences': 2,
    'max_new_tokens': 145,
}
generator: SubgoalGenerator = TransformerSubgoalGenerator(
    BartForConditionalGeneration.from_pretrained,
    path_to_generator_weights,
    env,
    subgoal_generation_kwargs,
)
generator.construct_network()

current_state2 = next(dataloader).cpu().numpy()[0]

nodes = [
    SearchTreeNode(
        state=current_state,
        value=0.0,
        low_level_path=[],
        parent_node=None,
        next_expand_with_k_generator=8,
    ),
    SearchTreeNode(
        state=current_state2,
        value=0.0,
        low_level_path=[],
        parent_node=None,
        next_expand_with_k_generator=8,
    ),
]

subgoals_0 = generator.get_subgoals(nodes[0])
subgoals_1 = generator.get_subgoals(nodes[1])

for input_node, subgoals in zip(nodes, [subgoals_0, subgoals_1]):
    fig = env.many_states_to_repr(
        [input_node.state, *[subgoal.state for subgoal in subgoals]],
        ['current_state', *[f'subgoal_proposition_{i}' for i in range(len(subgoals))]],
    )

    display(fig)



In [ ]:
generator.get_network()

# CLLP with Goal Generator - evaluation

In [ ]:
from importlib import reload
import carl.inference_components.cllp_utils as cllp_utils

reload(cllp_utils)

## With generator-generated goals

In [ ]:
display(env.many_states_to_repr([node.state for node in nodes], ['state_0', 'state_1']))
subgoals_first = [subgoal.state for subgoal in subgoals_0]
subgoals_second = [subgoal.state for subgoal in subgoals_1]
env.many_states_to_repr([*subgoals_first, *subgoals_second], ['subgoals_0' for _ in range(len(subgoals_first))] + ['subgoals_1' for _ in range(len(subgoals_second))])

In [ ]:
results = []
for input_node, subgoal_predictions in zip(nodes, [subgoals_0, subgoals_1]):
    cllp_ver_result = cllp_utils.verify_cllp_reaches_subgoals_from_initial_state(
        cllp=cllp,
        goals=[subgoal.state for subgoal in subgoal_predictions],
        initial_state=input_node.state,
        env_creation_fn=lambda: env,
        max_radius=10,
        add_first_batch_to_node_computations=True,
    )
    results.append(cllp_ver_result)
results

In [ ]:
from carl.inference_components.validator import BasicValidator


validator = BasicValidator(env, cllp, 10)

for input_node, result, subgoals in zip(nodes, results, [subgoals_0, subgoals_1]):
    logger.info(f'Verification result for node: {result}')
    for i in range(len(result.paths)):
        trajectory = cllp_utils.trajectory_from_actions(env, input_node.state, result.paths[i])
        subgoal_state = subgoals[i].state
        display(env.many_states_to_repr([input_node.state, subgoal_state, *trajectory],
                                        ['initial_state', 'subgoal', *[f'step_{i}' for i in range(len(trajectory))]]))    

## With small, randomly generated goals

In [ ]:
initial_state = next(dataloader).cpu().numpy()[0]

In [ ]:
def get_goals(
    gen_env: SokobanEnv, gen_initial_state: np.ndarray, num_goals: int = 5, steps_to_goal: int = 7
) -> list[np.ndarray]:
    out_goals = []
    for _ in range(num_goals):
        gen_state = gen_initial_state.copy()
        for _ in range(steps_to_goal):
            action = np.random.randint(0, 4)
            gen_env.set_state(gen_state)
            gen_state, _, done, _ = gen_env.step(action)
            if done:
                break
        out_goals.append(gen_state.copy())
    return out_goals


random_small_goals = get_goals(
    gen_env=env, gen_initial_state=initial_state, num_goals=10, steps_to_goal=3
)

In [ ]:
env.many_states_to_repr(
    [initial_state] + random_small_goals[:5],
    titles=['initial'] + [f'goal_{i}' for i in range(len(random_small_goals[:5]))],
)

In [ ]:
cllp_utils.verify_cllp_reaches_subgoals_from_initial_state(
    cllp=cllp,
    goals=random_small_goals,
    initial_state=initial_state,
    env_creation_fn=lambda: env,
    max_radius=7,
    add_first_batch_to_node_computations=True,
)

----

In [ ]:
# NPuzzle components

from carl.environment.n_puzzle.env import NPuzzleEnv
from carl.environment.n_puzzle.tokenizer import NPuzzleTokenizer

npuzzle_env = NPuzzleEnv(
    tokenizer=NPuzzleTokenizer(size_of_board=(5, 5))
)
npuzzle_env

In [ ]:
npuzzle_env

In [ ]:
import joblib as jl
datapath = './rl-data/validation/npuzzle/progress/fin/fin_puzzles_1000.pkl'
data = jl.load(datapath)

npuzzle_env.state_to_repr(data[0])

In [ ]:
from carl.environment.env import RepresentationType


npuzzle_env.restore_full_state_from_np_array_version(data[0])
npuzzle_env.state_to_repr(npuzzle_env.get_state(), repr_type=RepresentationType.GO_FIGURE)

In [ ]:
from carl.utils.notebook import instantiate_algorithm_from_grid

algo = instantiate_algorithm_from_grid('npuzzle_ada_solve', config_path='../../configs/solve/npuzzle', grid_entry_idx=0)

In [ ]:
from carl.solver.nodes import SearchTreeNode
planner = algo.solver.planner_class(data[0])
planner

In [ ]:
node_k8 = planner.get()
node_k4 = planner.get()
node_others = planner.get()
assert node_others is None

In [ ]:
node_k8.next_expand_with_k_generator, node_k4.next_expand_with_k_generator

In [ ]:
algo.solver.construct_networks()

In [ ]:
algo.solver.subgoal_generator.generator_k_list

In [ ]:
print(algo.solver.subgoal_generator.subgoal_generators[8].sub_generator)

In [ ]:
npuzzle_env.state_to_repr(algo.solver.subgoal_generator.get_subgoals(node_k8)[0].state, repr_type=RepresentationType.GO_FIGURE)

In [ ]:
npuzzle_env.state_to_repr(algo.solver.subgoal_generator.get_subgoals(node_k4)[0].state, repr_type=RepresentationType.GO_FIGURE)

In [ ]:
npuzzle_env.restore_full_state_from_np_array_version(data[0])
npuzzle_env.state_to_repr(npuzzle_env.get_state(), repr_type=RepresentationType.GO_FIGURE)

------

In [ ]:
# Rubik'

from carl.utils.notebook import instantiate_algorithm_from_grid

algo = instantiate_algorithm_from_grid('rubik_ada_solve', config_path='../../configs/solve/rubik', grid_entry_idx=0)


In [ ]:
import joblib

ds = joblib.load('./rl-data/validation/rubik/progress/shuffle_general/rubik_eval_data_part_shuffle_1000_0.pkl')
ds[:3]

In [ ]:
algo.solver.construct_networks()

In [1]:
root_state = 'gbbgywbwoyorybgrogyowrrbrrobryoggwgworoyobrbbyygywwwwg'

In [ ]:

root_node = algo.solver.planner_class(root_state)
node0 = root_node.get()
node1 = root_node.get()

In [ ]:
subgoal0 = algo.solver.subgoal_generator.get_subgoals(node0)

In [ ]:
subgoals1 = algo.solver.subgoal_generator.get_subgoals(node1)

In [2]:
from carl.environment.gym_rubik.rubik_env import RubikEnv
from carl.environment.gym_rubik.tokenizer import RubikCubeTokenizer
rubik_env = RubikEnv(RubikCubeTokenizer())

state_before_set = rubik_env.get_state()
state_before_set

'yyyyyyyyybbbbbbbbbrrrrrrrrrgggggggggooooooooowwwwwwwww'

In [3]:
rubik_env.restore_full_state_from_np_array_version(root_state)
state_after_set = rubik_env.get_state()
state_after_set

'bggwybowbryyoboggrrryrroobwwobggrwgyryoborbbowyywwygwg'

In [ ]:
action = algo.solver.validator.cllp.get_action(node0, subgoal0)